# Day 1. Vector / Matrix / Tensor / Shape

**오늘의 목표**
- Scalar, Vector, Matrix, Tensor가 무엇인지 설명할 수 있다
- `shape`를 보고 데이터가 어떻게 생겼는지 머릿속에 그릴 수 있다
- LLM 안에서 데이터가 `(batch, seq_len, n_embd)` 모양으로 흘러간다는 걸 안다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하면서 출력값을 직접 확인하세요.
> `# TODO` 표시가 있는 셀은 직접 채워보는 연습 문제입니다.

## 1. Scalar → Vector → Matrix → Tensor

| 이름 | 생김새 | 차원 수 (ndim) | 예시 |
|---|---|---|---|
| Scalar | 숫자 하나 | 0 | `175` |
| Vector | 숫자를 한 줄로 묶음 | 1 | `[175, 70, 25]` |
| Matrix | Vector를 여러 줄 쌓음 | 2 | 사람 3명의 `[키, 몸무게, 나이]` |
| Tensor | Matrix를 또 쌓음 (일반화) | 3 이상 | 문장 여러 개 × 단어 × 벡터 |

사실 PyTorch에서는 **전부 다 Tensor**라고 부릅니다. 차원 수만 다를 뿐이에요.

### 1-1. Python list로 먼저 표현해보기

In [ ]:
scalar = 175
vector = [175, 70, 25]           # 한 사람: [키, 몸무게, 나이]
matrix = [
    [175, 70, 25],               # 사람 A
    [160, 52, 31],               # 사람 B
    [182, 80, 28],               # 사람 C
]

print(scalar)
print(vector)
print(matrix)
print("사람 B의 몸무게:", matrix[1][1])

Python list도 표현은 되지만, 행렬 곱 같은 연산을 하려면 반복문을 직접 짜야 하고 느립니다.
그래서 **NumPy**나 **PyTorch**를 씁니다.

### 1-2. PyTorch Tensor로 바꾸기

In [ ]:
import torch

s = torch.tensor(175.0)
v = torch.tensor([175.0, 70.0, 25.0])
m = torch.tensor([
    [175.0, 70.0, 25.0],
    [160.0, 52.0, 31.0],
    [182.0, 80.0, 28.0],
])

for name, t in [("scalar", s), ("vector", v), ("matrix", m)]:
    print(f"{name:7s} | ndim={t.ndim} | shape={tuple(t.shape)}")

**`shape` 읽는 법**
- scalar: `()` → 축이 없음
- vector: `(3,)` → 숫자 3개짜리 한 줄
- matrix: `(3, 3)` → **(행 개수, 열 개수)** = (사람 3명, 특징 3개)

> 💡 shape의 **마지막 숫자**는 "가장 안쪽 리스트에 숫자가 몇 개인가"입니다.

## 2. Tensor (3차원 이상)

Matrix를 여러 개 쌓으면 3차원 Tensor가 됩니다.

예: 반 2개, 반마다 학생 3명, 학생마다 특징 4개 → shape `(2, 3, 4)`

In [ ]:
t = torch.arange(24).reshape(2, 3, 4)   # 0~23 숫자를 (2, 3, 4) 모양으로
print(t)
print("shape:", t.shape)
print("ndim :", t.ndim)
print("전체 원소 개수:", t.numel(), "= 2 * 3 * 4")

**인덱싱으로 감 잡기**: 바깥 축부터 하나씩 벗겨낸다고 생각하세요.

In [ ]:
print("t[0]        →", tuple(t[0].shape), "  # 첫 번째 반 (학생 3 × 특징 4)")
print("t[0][1]     →", tuple(t[0][1].shape), "    # 첫 번째 반의 두 번째 학생")
print("t[0][1][2]  →", t[0][1][2].item(), "      # 그 학생의 세 번째 특징 (scalar)")
print("t[:, 0, :]  →", tuple(t[:, 0, :].shape), "  # 모든 반의 첫 번째 학생")

## 3. Shape 다루기: `reshape`, `unsqueeze`, `transpose`

Transformer 코드에서 가장 자주 보는 shape 조작 세 가지입니다.
원소 개수는 그대로, **모양만** 바뀝니다.

In [ ]:
x = torch.arange(12)
print("원래        :", tuple(x.shape))
print("reshape(3,4):", tuple(x.reshape(3, 4).shape))
print("reshape(2,6):", tuple(x.reshape(2, 6).shape))
print("reshape(-1,4):", tuple(x.reshape(-1, 4).shape), " # -1은 '알아서 맞춰줘'")

m = x.reshape(3, 4)
print()
print("unsqueeze(0):", tuple(m.unsqueeze(0).shape), " # 맨 앞에 축 1개 추가 (batch 축 만들 때 자주 씀)")
print("transpose   :", tuple(m.transpose(0, 1).shape), "   # 행과 열 뒤집기 → Attention의 Kᵀ")

## 4. LLM과의 연결: `(B, T, C)`

GPT 코드에서 거의 모든 텐서는 이 모양입니다.

| 기호 | 의미 | 예 |
|---|---|---|
| `B` | batch: 한 번에 처리하는 문장 개수 | 2 |
| `T` | time / seq_len: 문장 안의 Token 개수 | 4 |
| `C` | channel / n_embd: Token 하나를 표현하는 벡터 길이 | 8 |

```text
"나는 오늘 학교에 간다"  →  Token 4개  →  각 Token이 길이 8짜리 벡터
                                       →  shape (4, 8)
문장 2개를 한 번에 넣으면                →  shape (2, 4, 8)
```

In [ ]:
B, T, C = 2, 4, 8
x = torch.randn(B, T, C)      # 랜덤 숫자로 채운 가짜 'Embedding 결과'

print("x.shape:", x.shape)
print("첫 번째 문장         :", tuple(x[0].shape))
print("첫 문장의 두 번째 Token:", tuple(x[0, 1].shape))
print()
print("첫 문장 두 번째 Token의 벡터 →", x[0, 1])

위에 찍힌 숫자 8개가 바로 커리큘럼에 나온 `"고양이" → [0.21, -0.53, 0.87, ...]` 같은 것입니다.
지금은 랜덤이지만, 학습이 끝나면 **의미를 담은 숫자**가 됩니다.

## 5. 연습 문제

각 셀의 `# TODO`를 채우고 실행해보세요. `assert`가 에러 없이 통과하면 정답입니다.

**Q1.** 아래 텐서의 shape를 실행하기 **전에** 예측해서 `answer`에 적어보세요.

In [ ]:
a = torch.zeros(5, 3)
b = torch.zeros(2, 5, 3)
c = torch.zeros(7)

answer_a = None  # TODO: 예) (5, 3)
answer_b = None  # TODO
answer_c = None  # TODO

assert tuple(a.shape) == answer_a, f"a는 {tuple(a.shape)}"
assert tuple(b.shape) == answer_b, f"b는 {tuple(b.shape)}"
assert tuple(c.shape) == answer_c, f"c는 {tuple(c.shape)}"
print("Q1 정답!")

**Q2.** 0~23 숫자를 담은 1차원 텐서를 `(B=2, T=3, C=4)` 모양으로 바꾸세요.

In [ ]:
x = torch.arange(24)
y = None  # TODO: x를 reshape

assert tuple(y.shape) == (2, 3, 4)
print("Q2 정답!", y.shape)

**Q3.** `(T=4, C=8)` 모양의 문장 하나에 batch 축을 추가해서 `(1, 4, 8)`로 만드세요.

In [ ]:
sentence = torch.randn(4, 8)
batched = None  # TODO

assert tuple(batched.shape) == (1, 4, 8)
print("Q3 정답!", batched.shape)

**Q4.** `(B, T, C) = (2, 4, 8)` 텐서에서 **두 번째 문장의 마지막 Token 벡터**를 꺼내세요. 결과 shape는?

In [ ]:
x = torch.randn(2, 4, 8)
last_token = None  # TODO

assert tuple(last_token.shape) == (8,)
assert torch.equal(last_token, x[1][3])
print("Q4 정답!", last_token.shape)

> 💡 Q4가 사실 GPT의 **텍스트 생성**에서 쓰는 동작입니다.
> 다음 Token을 예측할 때는 문장의 **마지막 위치**의 출력만 꺼내서 씁니다. (`logits[:, -1, :]`)

## 6. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요. 피드백 드릴게요.

1. 벡터와 행렬의 차이는 무엇인가?
2. `shape`가 `(2, 4, 8)`인 텐서를 LLM 관점에서 설명해보세요. 각 숫자는 무엇을 의미하나요?
3. `reshape`를 해도 바뀌지 않는 것은 무엇인가?
4. (미리보기) `(3, 4)` 행렬과 `(4, 5)` 행렬을 곱하면 결과 shape는? → 내일 Day 2에서 확인합니다.

**내 답:**

1.
2.
3.
4.